In [0]:
%sql

-- Null Check
select 
    count(*) As Total_rows,
    SUM(case when customer_id is NULL then 1 else 0 END) as Null_customer_id,
    SUM(case when merchant_id is NULL then 1 else 0 END) as Null_merchant_id,
    SUM(case when transaction_timestamp is NUll then 1 else 0 END) as Null_transaction_timestamp,
    SUM(case when amount is NULL then 1 else 0 end) as NULL_amount
from fintech_fraud_risk.silver.transactions;

-- Check Deduplicate
select transaction_id, count(*) as cnt from fintech_fraud_risk.silver.transactions
Group by transaction_id
having count(*) > 1;


In [0]:
%sql
-- invalid format

select
    sum(case when amount <=0 Then 1 else 0 END) as invalid_amount,
    sum(case when transaction_status Not in ('Success','Failed','Pending') Then 1 else 0 end ) as invalid_status,
    sum(case when try_cast(transaction_timestamp as Timestamp) is null then 1 else 0 end ) as bad_timestamp    

from fintech_fraud_risk.silver.transactions;

In [0]:
%sql
-- 4. ORPHAN FOREIGN KEY CHECK — anti-join pattern
SELECT COUNT(*) AS orphan_customer_ids 
FROM fintech_fraud_risk.bronze.bronze_transactions t
LEFT ANTI JOIN fintech_fraud_risk.bronze.bronze_customers c
  ON t.customer_id = c.customer_id;

In [0]:
from pyspark.sql import functions as F

def profile_table(df, id_col, name, fk_checks=None):
    total = df.count()
    print(f"\n=== {name} ({total:,} rows) ===")
    for c in df.columns:
        n = df.filter(F.col(c).isNull()).count()
        if n > 0:
            print(f"  NULL  {c:20s} {n:>7,}  ({n/total*100:.2f}%)")
    dup = total - df.dropDuplicates([id_col]).count()
    print(f"  DUP   {id_col:20s} {dup:>7,}  ({dup/total*100:.2f}%)")
    if fk_checks:
        for fk_col, valid_df, valid_col in fk_checks:
            orphan = df.join(valid_df.select(valid_col), df[fk_col] == valid_df[valid_col], "left_anti").count()
            print(f"  FK    {fk_col:20s} {orphan:>7,}  ({orphan/total*100:.2f}%) orphan")

bronze_customers = spark.table("fintech_fraud_risk.bronze.bronze_customers")
customers = spark.table("fintech_fraud_risk.silver.customers")
bronze_merchants = spark.table("fintech_fraud_risk.bronze.bronze_merchants")
bronze_devices   = spark.table("fintech_fraud_risk.bronze.bronze_devices")
bronze_login     = spark.table("fintech_fraud_risk.bronze.bronze_login_events")
bronze_fraud     = spark.table("fintech_fraud_risk.bronze.bronze_fraud_alerts")
bronze_cb        = spark.table("fintech_fraud_risk.bronze.bronze_chargebacks")

all_txns = spark.table("fintech_fraud_risk.bronze.bronze_transactions").select("transaction_id") \
    .union(spark.table("fintech_fraud_risk.bronze.bronze_transaction_cdc").select("transaction_id")).distinct()

profile_table(bronze_customers, "customer_id", "customers")
profile_table(customers, "customer_id", "customers")
profile_table(bronze_merchants, "merchant_id", "merchants")
profile_table(bronze_devices, "device_id", "devices", [("customer_id", bronze_customers, "customer_id")])
profile_table(bronze_login, "login_id", "login_events",
              [("customer_id", bronze_customers, "customer_id"), ("device_id", bronze_devices, "device_id")])
profile_table(bronze_fraud, "alert_id", "fraud_alerts", [("transaction_id", all_txns, "transaction_id")])
profile_table(bronze_cb, "chargeback_id", "chargebacks", [("transaction_id", all_txns, "transaction_id")])